In [ ]:
import kan
import torch
import matplotlib.pyplot as plt

if __name__ == "__main__":

    f = lambda x: torch.exp(torch.sin(torch.pi*x[:,[0]]) + x[:,[1]]**2)
    dataset = kan.utils.create_dataset(f, n_var=2)

    model = kan.KAN(width=[2, 5, 1], grid=3, k=3, seed=42)

    model(dataset['train_input'])
    model.plot()

    model.fit(dataset, opt="LBFGS", lamb=0.001)
    model.plot()

    model = model.prune()
    model.plot()
    model.fit(dataset, opt="LBFGS", steps=50)
    model = model.refine(10)
    model.fit(dataset, opt="LBFGS", steps=50)

    model.auto_symbolic()
    model.fit(dataset, opt="LBFGS", lamb=50)

    sym_exp = kan.utils.ex_round(model.symbolic_formula()[0][0], 4)
    print(sym_exp)
    plt.show()

In [ ]:
import torch
import kan
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)

# ======================================
# LOAD GT DATASETS
# ======================================

r1_gt = pd.read_csv("r1_gt_imu.csv")
r2_gt = pd.read_csv("r2_gt_imu.csv")
r3_gt = pd.read_csv("r3_gt_imu.csv")

gt_data = pd.concat(
    [r1_gt, r2_gt, r3_gt],
    ignore_index=True
)

# ======================================
# LOAD IMU TXT DATASETS
# ======================================

imu_cols = [
    "timestamp",
    "wx", "wy", "wz",
    "ax", "ay", "az"
]

r1_imu = pd.read_csv(
    "r1_imu.txt",
    sep=r"\s+",
    comment="#",
    names=imu_cols
)

r2_imu = pd.read_csv(
    "r2_imu.txt",
    sep=r"\s+",
    comment="#",
    names=imu_cols
)

r3_imu = pd.read_csv(
    "r3_imu.txt",
    sep=r"\s+",
    comment="#",
    names=imu_cols
)

imu_data = pd.concat(
    [r1_imu, r2_imu, r3_imu],
    ignore_index=True
)

# ======================================
# CHECK LENGTHS
# ======================================

print("GT samples :", len(gt_data))
print("IMU samples:", len(imu_data))

# Only use this if row counts match
min_len = min(len(gt_data), len(imu_data))

gt_data = gt_data.iloc[:min_len].reset_index(drop=True)
imu_data = imu_data.iloc[:min_len].reset_index(drop=True)

# ======================================
# MERGE DATA
# ======================================

data = pd.concat(
    [gt_data, imu_data],
    axis=1
)

print("Total Samples:", len(data))

# ======================================
# FEATURES & TARGET
# ======================================

feature_names = [
    'tx', 'ty', 'tz',
    'wx', 'wy', 'wz',
    'ax', 'ay', 'az'
]

output_name = 'qw'

X_np = data[feature_names].values
y_np = data[output_name].values

print("Features Shape:", X_np.shape)
print("Target Shape:", y_np.shape)

In [ ]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=SEED)

y_train_np = y_train_np.reshape(-1, 1)
y_test_np = y_test_np.reshape(-1, 1)

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = torch.tensor(scaler_x.fit_transform(X_train_np))
X_test = torch.tensor(scaler_x.transform(X_test_np))
y_train = torch.tensor(scaler_y.fit_transform(y_train_np))
y_test = torch.tensor(scaler_y.transform(y_test_np))

dataset = {
    "train_input": X_train,
    "train_label": y_train,  # Changed from 'train_output' to 'train_label'
    "test_input": X_test,
    "test_label": y_test    # Changed from 'test_output' to 'test_label'
}

In [ ]:
model = kan.KAN(width=[3, 5, 1], grid=3, k=3, seed=SEED)

model.fit(dataset, opt="LBFGS", steps=20,
          lamb=1.89e-4, lamb_entropy=8.209, reg_metric="edge_forward_spline_u")


model.plot()

#model.fit(dataset, opt="LBFGS", lamb=0.001)

model= model.prune()
model.plot()

model.fit(dataset, opt="LBFGS", steps=20)# lr=1.0
#model = model.refine(10)
#model.fit(dataset, opt="LBFGS", steps=50)
plt.show()
